# 07 · Packed towers and mass transfer

Packed columns have no stages: gas and liquid meet continuously over the packing, and their height is set by
rates, not equilibrium alone. The transfer-unit method splits the height into a number of transfer units
(how hard the separation is) and a height of a transfer unit (how good the packing and the mass-transfer
coefficients are). This notebook computes both, relates overall to film coefficients through the two-film
theory, and estimates diffusivities and mass-transfer coefficients from correlations.

**What you will learn**
- compute NTU by integration and by Colburn's equation
- size a packed tower from HTU and NTU
- use two-film theory to identify the controlling resistance
- estimate diffusivities and coefficients from correlations

**Before you start:** notebook 06  ·  **Time:** about 60 min

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engsep import absorption as ab
from engsep import transport as tr
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## NTU: the integral

In [2]:
m, y_in, y_out, x_in = 0.8, 0.02, 0.002, 0.0
for A in (1.2, 1.5, 2.0, 3.0):
    L_over_V = A * m
    x_of_y = lambda y, L_over_V=L_over_V: x_in + (y - y_out) / L_over_V
    N_num = ab.ntu_gas(y_in, y_out, lambda x: m * x, x_of_y)
    N_col = ab.ntu_colburn(y_in, y_out, m, x_in, A)
    print(f"A = {A}: N_OG = {N_num:.3f} (integral), {N_col:.3f} (Colburn)")

A = 1.2: N_OG = 5.498 (integral), 5.498 (Colburn)
A = 1.5: N_OG = 4.159 (integral), 4.159 (Colburn)
A = 2.0: N_OG = 3.410 (integral), 3.409 (Colburn)
A = 3.0: N_OG = 2.919 (integral), 2.919 (Colburn)


The number of overall gas-phase transfer units is $\int dy/(y - y^*)$ along the column: the driving force
$y - y^*$ shrinks as the operating and equilibrium lines approach, and the integral grows. For a linear
equilibrium line Colburn's closed form agrees with the integral to four figures - and, like the Kremser
equation, it says that more solvent (a larger $A$) means fewer transfer units.

## HTU: the packing and the coefficients

In [3]:
G_M = 0.05                                                           # kmol/(m2 s) of gas
for packing, K_y_a in (("25 mm Raschig rings", 0.03), ("25 mm Pall rings", 0.05), ("structured packing", 0.08)):
    H_OG = ab.htu_gas(G_M, K_y_a)
    print(f"{packing:<22}: K_y a = {K_y_a} kmol/(m3 s) -> H_OG = {H_OG:.2f} m -> height for N_OG = {ab.ntu_colburn(y_in, y_out, m, x_in, 1.5):.2f}: {ab.packed_height(H_OG, ab.ntu_colburn(y_in, y_out, m, x_in, 1.5)):.1f} m")

25 mm Raschig rings   : K_y a = 0.03 kmol/(m3 s) -> H_OG = 1.67 m -> height for N_OG = 4.16: 6.9 m
25 mm Pall rings      : K_y a = 0.05 kmol/(m3 s) -> H_OG = 1.00 m -> height for N_OG = 4.16: 4.2 m
structured packing    : K_y a = 0.08 kmol/(m3 s) -> H_OG = 0.62 m -> height for N_OG = 4.16: 2.6 m


The height of a transfer unit is the gas flux over the volumetric mass-transfer coefficient: a property of
the packing and the flows, measured by the packing vendor, typically 0.3-1 m. Height = HTU x NTU separates
what the chemistry demands from what the hardware delivers.

## Two-film theory

In [4]:
for m_ in (0.05, 0.5, 5.0, 50.0):
    tf = ab.overall_from_film(k_y=0.1, k_x=1.0, m=m_)
    print(f"m = {m_:5.2f}: K_y = {tf['K_y']:.4f}, fraction of the resistance in the gas film {tf['gas_resistance_fraction']:.0%}")

m =  0.05: K_y = 0.0995, fraction of the resistance in the gas film 100%
m =  0.50: K_y = 0.0952, fraction of the resistance in the gas film 95%
m =  5.00: K_y = 0.0667, fraction of the resistance in the gas film 67%
m = 50.00: K_y = 0.0167, fraction of the resistance in the gas film 17%


The overall resistance is the sum of the two film resistances, $1/K_y = 1/k_y + m/k_x$, with the equilibrium
slope weighting the liquid side. A very soluble gas (small $m$: ammonia in water) is gas-film controlled;
a sparingly soluble one (large $m$: oxygen, CO$_2$) is liquid-film controlled. Knowing which film controls
tells you what to improve - gas velocity or liquid turbulence.

## Diffusivities from correlations

In [5]:
rows = []
for gas in ("water", "CO2", "NH3", "SO2", "benzene"):
    rows.append((gas, tr.fuller(gas, "air", 298.15) * 1e4))
print("gas diffusivities in air at 25 degC (Fuller), cm2/s:", {g: round(d, 3) for g, d in rows}, "- measured: water 0.26, CO2 0.16, NH3 0.28")
D_l = tr.wilke_chang(18.02, 2.6, 298.15, 0.89, 34.0)                # CO2 in water (V = 34 cm3/mol)
print(f"CO2 in water at 25 degC (Wilke-Chang): {D_l:.2e} m2/s (measured 1.9e-9)")

gas diffusivities in air at 25 degC (Fuller), cm2/s: {'water': 0.253, 'CO2': 0.158, 'NH3': 0.245, 'SO2': 0.127, 'benzene': 0.09} - measured: water 0.26, CO2 0.16, NH3 0.28
CO2 in water at 25 degC (Wilke-Chang): 2.05e-09 m2/s (measured 1.9e-9)


Gas diffusivities are around 0.1-0.3 cm$^2$/s and liquid ones ten thousand times smaller - the reason
liquid films control for sparingly soluble gases and why liquid-side turbulence is worth engineering. The
correlations are good to 5-10 % in gases and 10-25 % in liquids: adequate for design, not for science.

## Mass-transfer coefficients from correlations

In [6]:
D_g, nu_g, d_p = tr.fuller("CO2", "air", 298.15), 1.6e-5, 0.025
Sc = nu_g / D_g
for u in (0.5, 1.0, 2.0):
    Re = u * d_p / nu_g
    Sh = tr.sherwood_sphere(Re, Sc)
    print(f"u = {u} m/s over 25 mm packing: Re = {Re:.0f}, Sc = {Sc:.2f}, Sh = {Sh:.1f}, k_c = {Sh * D_g / d_p * 100:.2f} cm/s")
print(f"penetration theory for a liquid film exposed 0.1 s: k_L = {tr.penetration_coefficient(1.9e-9, 0.1) * 100:.3f} cm/s")

u = 0.5 m/s over 25 mm packing: Re = 781, Sc = 1.01, Sh = 18.9, k_c = 1.19 cm/s
u = 1.0 m/s over 25 mm packing: Re = 1563, Sc = 1.01, Sh = 25.8, k_c = 1.63 cm/s
u = 2.0 m/s over 25 mm packing: Re = 3125, Sc = 1.01, Sh = 35.7, k_c = 2.25 cm/s
penetration theory for a liquid film exposed 0.1 s: k_L = 0.016 cm/s


Sherwood correlations of the form $Sh = a + b\,Re^{1/2}Sc^{1/3}$ give the gas-side coefficient from the flow;
Higbie's penetration theory gives the liquid side from the time a liquid element is exposed as it trickles
over a packing piece. Both feed $K_y a$, and hence the HTU, from first principles when vendor data are
missing.

## Inside the algorithm: the transfer-unit integral

In [7]:
from scipy import integrate
A_ = 1.5
val, _ = integrate.quad(lambda y: 1 / (y - m * (x_in + (y - y_out) / (A_ * m))), y_out, y_in)
print("by hand: N_OG =", val, "; Colburn:", ab.ntu_colburn(y_in, y_out, m, x_in, A_))

by hand: N_OG = 4.158883083359672 ; Colburn: 4.1588830833596715


## Exercises
1. A packed tower absorbs acetone (2 %) from air into water (m = 2.0) with 95 % recovery at A = 1.6; G_M = 0.04 kmol/(m2 s), K_y a = 0.06 kmol/(m3 s). Find the height. Then halve the gas flux (a wider tower): how does the height change, and why less than proportionally (assume K_y a ~ G^0.7)?
2. For a gas with m = 5 and k_y = 0.1, k_x = 0.5, what fraction of the resistance is in the liquid? By how much must k_x be raised (more liquid turbulence) to halve the total resistance?
3. **Implement it yourself:** compute Fuller's diffusivity for ammonia in air from the diffusion volumes and molar masses in `transport.FULLER_V` and `transport.MOLAR_MASS`, and compare with the library and with the measured 0.28 cm2/s.